# Baseline — PaddleOCR gốc

Notebook này đánh giá mô hình PaddleOCR gốc, chưa fine-tune, trên tập `rec_test.txt`. Kết quả là mốc đối chiếu cho các notebook fine-tune.

**Chuẩn bị trên Kaggle:** gắn Input `baonguyenngoc0902/hcmus-nlp-vietnamese-recognition-resources`, bật **GPU** và **Internet**. Internet chỉ dùng để cài thư viện và tải model PaddleOCR gốc; dữ liệu đầu vào không bị thay đổi. Notebook này tự chứa, không cần clone repository.

## 1. Cài đặt môi trường

Cell dưới đây cài PaddlePaddle GPU 3.2.1 (bản build CUDA 12.6) từ file wheel tải riêng bằng `pip download`, vì file nặng khoảng 1,6 GB và tải thẳng bằng `pip` dễ bị đứt giữa chừng. Wheel được lưu ở `/kaggle/temp` nên không làm phình Output. PaddleOCR 3.3.0 và PaddleX 3.3.0 được ghim cố định cùng Paddle 3.2.1 để baseline và cả hai lần fine-tune chạy trong một môi trường duy nhất, nhờ đó kết quả tái lập được. Cell cũng gỡ `torch` có sẵn của Kaggle (Paddle hạ cấp một số thư viện NVIDIA làm `torch` hỏng) và `langchain` (bản mới của Kaggle làm `import paddleocr` báo `ModuleNotFoundError: langchain.docstore`); notebook này không dùng hai gói đó, rồi xác nhận phiên bản bằng `pip show` và `run_check`. Nếu cell báo lỗi thì không chạy các phần tiếp theo.

In [ ]:
import subprocess
import sys
import zipfile
from importlib.metadata import version
from pathlib import Path

PADDLE_VERSION = "3.2.1"
PADDLEOCR_VERSION = "3.3.0"
PADDLEX_VERSION = "3.3.0"  # PaddleOCR 3.3.0 yêu cầu paddlex>=3.3.0,<3.4.0
CUDA_TAG = "cu126"  # driver Kaggle (CUDA 12.x trở lên) chạy được bản build này

PADDLE_INDEX = f"https://www.paddlepaddle.org.cn/packages/stable/{CUDA_TAG}/"
WHEEL_DIR = Path("/kaggle/temp")  # thư mục tạm, không bị lưu vào Output
WHEEL_DIR.mkdir(parents=True, exist_ok=True)

# Dọn wheel cũ nếu từng tải nhầm vào /kaggle/working (làm phình Output)
for stale in Path("/kaggle/working").glob("paddlepaddle_gpu-*.whl"):
    stale.unlink()

subprocess.run(["nvidia-smi"], check=True)

# 1) Tải riêng file wheel bằng pip download (pip tự chọn đúng tên file theo phiên bản Python), kiểm tra file nguyên vẹn
for attempt in range(1, 4):
    for old in WHEEL_DIR.glob("paddlepaddle_gpu-*.whl"):
        old.unlink()
    subprocess.run(
        [sys.executable, "-m", "pip", "download", f"paddlepaddle-gpu=={PADDLE_VERSION}",
         "--no-deps", "-d", str(WHEEL_DIR), "-i", PADDLE_INDEX, "--timeout", "600", "--retries", "10"],
        check=True,
    )
    WHEEL_PATH = next(WHEEL_DIR.glob("paddlepaddle_gpu-*.whl"))
    try:
        with zipfile.ZipFile(WHEEL_PATH):  # file bị cắt cụt sẽ báo BadZipFile
            pass
        print(f"Wheel OK: {WHEEL_PATH} ({WHEEL_PATH.stat().st_size / 1e9:.2f} GB)")
        break
    except zipfile.BadZipFile:
        print(f"Lần {attempt}: wheel hỏng, xóa và tải lại")
        WHEEL_PATH.unlink(missing_ok=True)
else:
    raise RuntimeError("Không tải được wheel nguyên vẹn sau 3 lần.")

# 2) Cài Paddle từ file local (dependency lấy từ PyPI), rồi PaddleOCR, PaddleX và thư viện phụ
pip = [sys.executable, "-m", "pip", "install", "--timeout", "600", "--retries", "10"]
subprocess.run(pip + [str(WHEEL_PATH)], check=True)
subprocess.run(
    pip + [
        f"paddleocr=={PADDLEOCR_VERSION}",
        f"paddlex=={PADDLEX_VERSION}",
        "python-Levenshtein>=0.25",
        "PyYAML>=6.0",
        "tqdm>=4.0",
    ],
    check=True,
)

# 2b) Gỡ các gói có sẵn của Kaggle mà notebook này không dùng nhưng gây lỗi:
#  - torch: Paddle hạ cấp nvidia-nccl/cudnn làm torch hỏng
#  - langchain: bản mới của Kaggle (1.x) không còn module langchain.docstore, làm `import paddleocr` báo ModuleNotFoundError
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio", "langchain", "langchain-community"],
    check=False,
)

# 3) Xác nhận: đúng phiên bản đã ghim, và Paddle chạy được trên GPU (tiến trình mới nên không cần restart)
subprocess.run(
    [sys.executable, "-m", "pip", "show", "paddlepaddle-gpu", "paddleocr", "paddlex"],
    check=True,
)
for package, expected in [
    ("paddlepaddle-gpu", PADDLE_VERSION),
    ("paddleocr", PADDLEOCR_VERSION),
    ("paddlex", PADDLEX_VERSION),
]:
    installed = version(package)
    assert installed == expected, f"{package}: cài {installed}, cần {expected}"
subprocess.run(
    [sys.executable, "-c",
     "import paddle; paddle.utils.run_check(); print(paddle.__version__, paddle.device.get_device())"],
    check=True,
)

## 2. Ghi nhận môi trường chạy và cố định seed

Cell này đặt seed 42 để giảm tính ngẫu nhiên, kiểm tra PaddlePaddle có nhận GPU hay không, rồi in phiên bản Python, PaddlePaddle, PaddleOCR, PaddleX và thông tin GPU. Thông tin này được lưu vào file metrics và cần chép vào `docs/experiment_log.md` sau khi chạy. Nếu vừa cài lại gói ở mục 1 mà cell này báo lỗi import, hãy **Restart session** rồi chạy lại từ mục 2 (không cần cài lại).

In [ ]:
import json
import os
import platform
import random
import subprocess
import sys
import time
import warnings
from importlib.metadata import version
from pathlib import Path

warnings.filterwarnings("ignore", message="No ccache found")  # cảnh báo vô hại của Paddle

# Đặt trước khi tạo model. BOS là nguồn model chính thức dự phòng khi Hugging Face không truy cập được.
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

import numpy as np
import paddle
import paddleocr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
paddle.seed(SEED)

assert paddle.is_compiled_with_cuda(), (
    "PaddlePaddle has no CUDA support. Enable a Kaggle GPU and rerun installation."
)

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    text=True,
).strip()

ENV_INFO = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "paddlepaddle": paddle.__version__,
    "paddleocr": paddleocr.__version__,
    "paddlex": version("paddlex"),
    "seed": SEED,
    "gpu": gpu_info.splitlines(),
}
print(json.dumps(ENV_INFO, ensure_ascii=False, indent=2))

## 3. Lấy code dùng chung từ repository

Hàm tính metric nằm trong `src/metrics.py` của repository và là bản chuẩn cho mọi lần đánh giá (baseline và fine-tune), nên notebook không chép lại mà clone repository về `/kaggle/working/repo` rồi import trực tiếp. Cell ghi lại mã commit đã dùng vào `ENV_INFO`, để biết số liệu trong báo cáo ứng với phiên bản code nào. Repository phải là public và đã push trước khi chạy cell này.

In [ ]:
REPO_URL = "https://github.com/nguyenngocbao/nlp-vietnamese-ocr-recognition.git"
REPO_REF = "main"  # có thể đổi thành mã commit hoặc tag để cố định hẳn phiên bản code
REPO_DIR = Path("/kaggle/working/repo")

if not (REPO_DIR / "src" / "metrics.py").is_file():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

ENV_INFO["repo_url"] = REPO_URL
ENV_INFO["repo_commit"] = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Repo commit:", ENV_INFO["repo_commit"])

## 4. Tìm dữ liệu

Kaggle gắn dữ liệu ở thư mục chỉ đọc `/kaggle/input` và đã giải nén sẵn thành thư mục `vi_rec_100k/` (chứa ba file nhãn và ảnh) cùng `vi_dict.txt`, nên không cần giải nén. Cell này tìm thư mục dataset (thử cả hai kiểu đường dẫn mà Kaggle dùng), rồi xác định đường dẫn thực tế của từ điển và ba file nhãn train, validation, test. Kết quả nhỏ được ghi vào `/kaggle/working/results`.

In [ ]:
DATASET_SLUG = "hcmus-nlp-vietnamese-recognition-resources"
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/baonguyenngoc0902") / DATASET_SLUG,  # kiểu mount hiện tại
    Path("/kaggle/input") / DATASET_SLUG,  # kiểu mount cũ
]
DATASET_DIR = next((path for path in DATASET_CANDIDATES if path.exists()), None)
assert DATASET_DIR is not None, (
    "Attach the Kaggle dataset in the Input panel. Đã thử: " + ", ".join(map(str, DATASET_CANDIDATES))
)

WORK_DIR = Path("/kaggle/working")
DATA_DIR = DATASET_DIR  # dữ liệu đã giải nén sẵn, đọc thẳng từ Input (chỉ đọc)
RESULTS_DIR = WORK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def find_file(name: str) -> Path:
    # Chỉ kiểm tra hai vị trí đã biết. Không dùng rglob vì thư mục chứa hàng trăm nghìn ảnh, quét rất chậm trên ổ Input
    candidates = [DATASET_DIR / name, DATASET_DIR / "vi_rec_100k" / name]
    matches = [path for path in candidates if path.is_file()]
    assert matches, f"Không thấy {name} trong: {candidates}"
    return matches[0]  # vị trí nông nhất được ưu tiên

DICT_PATH = find_file("vi_dict.txt")
TRAIN_LABELS = find_file("rec_train.txt")
VAL_LABELS = find_file("rec_val.txt")
TEST_LABELS = find_file("rec_test.txt")

print("dataset:", DATASET_DIR)
print("dict:", DICT_PATH, f"({len(DICT_PATH.read_text(encoding='utf-8').splitlines())} dòng)")
print("train labels:", TRAIN_LABELS)
print("val labels:", VAL_LABELS)
print("test labels:", TEST_LABELS)

## 5. Kiểm tra các file nhãn

Mỗi dòng nhãn có dạng `đường_dẫn_ảnh<TAB>văn_bản_đúng`. Cell dưới đây đọc ba split, kiểm tra số lượng dòng theo đề bài (93.997 train, 3.000 validation và 3.003 test), rồi hiển thị ba nhãn test đầu tiên. Nếu số lượng không đúng, cần dừng lại kiểm tra dữ liệu trước khi đánh giá model.

In [ ]:
def read_label_lines(path: Path) -> list[str]:
    lines = path.read_text(encoding="utf-8").splitlines()
    assert all("\t" in line for line in lines), f"Invalid label format in {path}"
    return lines

split_lines = {
    "train": read_label_lines(TRAIN_LABELS),
    "val": read_label_lines(VAL_LABELS),
    "test": read_label_lines(TEST_LABELS),
}
for split, lines in split_lines.items():
    print(f"{split}: {len(lines):,} labels")

assert len(split_lines["train"]) == 93_997
assert len(split_lines["val"]) == 3_000
assert len(split_lines["test"]) == 3_003
print("First three test labels:")
print("\n".join(split_lines["test"][:3]))


## 6. Ghép nhãn với ảnh test

File nhãn lưu đường dẫn tương đối của ảnh (ví dụ `test/xxx.jpg`, tính từ thư mục chứa file nhãn). Cell này đổi chúng thành đường dẫn ảnh thật và kiểm tra cả 3.003 ảnh test đều tồn tại. Chỉ `rec_test.txt` được dùng cho các số liệu baseline cuối cùng.

In [ ]:
def resolve_image_path(raw_path: str) -> Path:
    relative_path = Path(raw_path.replace("\\", "/").lstrip("./"))
    for candidate in (TEST_LABELS.parent / relative_path, DATA_DIR / relative_path):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Cannot resolve image path: {raw_path}")

test_examples = []
for line in split_lines["test"]:
    raw_path, gt = line.split("\t", 1)
    test_examples.append(
        {
            "image": raw_path.replace("\\", "/"),  # đường dẫn trong file nhãn, ghi vào JSONL
            "path": str(resolve_image_path(raw_path)),  # đường dẫn thật, dùng để dự đoán
            "gt": gt,
        }
    )

assert len(test_examples) == 3_003
assert all(Path(row["path"]).is_file() for row in test_examples)
print("Resolved test images:", len(test_examples))
print("Example:", test_examples[0])

## 7. Tải model baseline gốc

Baseline là model chính thức `latin_PP-OCRv5_mobile_rec` của PaddleOCR (nhóm chữ Latin, gần tiếng Việt hơn), chưa fine-tune. Đây cũng là model được fine-tune ở các notebook sau, nên bảng so sánh cuối cùng chỉ khác nhau ở bước fine-tune. Model này có trong PaddleX 3.3.0 nhưng không có trong 3.0.3. Cell chạy thử một ảnh đầu tiên để xác nhận cấu trúc kết quả trước khi dự đoán toàn bộ tập test.

In [ ]:
from paddleocr import TextRecognition

MODEL_NAME = "latin_PP-OCRv5_mobile_rec"
# Batch 1 để kết quả giống tools/infer_rec.py và không phụ thuộc padding theo batch.
BATCH_SIZE = 1

model_init_start = time.perf_counter()
recognizer = TextRecognition(
    model_name=MODEL_NAME,
    device="gpu:0",
)
model_init_seconds = time.perf_counter() - model_init_start

sample_result = recognizer.predict(
    input=[test_examples[0]["path"]],
    batch_size=1,
)[0]
sample_payload = sample_result.json
if isinstance(sample_payload, str):
    sample_payload = json.loads(sample_payload)

print(sample_payload)
assert "res" in sample_payload and "rec_text" in sample_payload["res"]

## 8. Dự đoán toàn bộ tập test và lưu JSONL

Cell này đưa ảnh test vào model với `BATCH_SIZE = 1`. Kiểm tra trên 200 ảnh cho thấy batch 64 làm 118/200 dự đoán khác batch 1 do padding theo batch, nên baseline dùng batch 1 (toàn bộ test chỉ ~37 giây). Mỗi kết quả được lưu dưới dạng một dòng JSON với ba trường bắt buộc: `image` (đường dẫn ảnh trong file nhãn), `gt` (nhãn đúng) và `pred` (dự đoán). File JSONL là đầu ra cần nộp cho baseline.

In [ ]:
from tqdm.auto import tqdm

def text_from_result(result) -> str:
    payload = result.json
    if isinstance(payload, str):
        payload = json.loads(payload)
    return str(payload["res"]["rec_text"])

prediction_start = time.perf_counter()
prediction_rows = []

for start in tqdm(range(0, len(test_examples), BATCH_SIZE), desc="Predicting test set"):
    batch = test_examples[start : start + BATCH_SIZE]
    results = recognizer.predict(
        input=[row["path"] for row in batch],
        batch_size=len(batch),
    )
    assert len(results) == len(batch), "Prediction count differs from input count."

    for example, result in zip(batch, results):
        prediction_rows.append(
            {"image": example["image"], "gt": example["gt"], "pred": text_from_result(result)}
        )

prediction_seconds = time.perf_counter() - prediction_start
assert len(prediction_rows) == len(test_examples)

PREDICTIONS_PATH = RESULTS_DIR / "pred_baseline_ppocrv5_mobile.jsonl"
with PREDICTIONS_PATH.open("w", encoding="utf-8") as file:
    for row in prediction_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

# Đọc lại để xác nhận file hợp lệ: đủ số dòng và đủ ba trường image/gt/pred
with PREDICTIONS_PATH.open(encoding="utf-8") as file:
    saved_rows = [json.loads(line) for line in file if line.strip()]
assert len(saved_rows) == len(prediction_rows)
assert all({"image", "gt", "pred"} <= row.keys() for row in saved_rows)

print(f"Prediction time: {prediction_seconds:.2f} seconds")
print("Predictions:", PREDICTIONS_PATH)

## 9. Tính metric báo cáo

Mọi so sánh đều chuẩn hóa Unicode về NFC trước (tiếng Việt có hai cách mã hóa cùng một chữ). Các metric được tính trên cùng tập `rec_test.txt`:

- `acc` và `norm_edit_dis`: số liệu chính, **giữ khoảng trắng**. `acc` là tỷ lệ dự đoán khớp hoàn toàn với nhãn; `norm_edit_dis` là độ tương đồng trung bình dựa trên khoảng cách Levenshtein đã chuẩn hóa, càng gần 1 càng tốt.
- `acc_ignore_space` và `norm_edit_dis_ignore_space`: số liệu phụ, **bỏ khoảng trắng**, để đối chiếu với các cách đo bỏ khoảng trắng.

Metric được tính bằng `src/metrics.py` của repository (đã clone ở mục 3), nên sau khi tải file JSONL về chạy `python eval.py --predictions results/pred_baseline_ppocrv5_mobile.jsonl` để đối chiếu. Cell cũng lưu phiên bản thư viện, seed, GPU, thời gian khởi tạo model và thời gian dự đoán để đưa vào báo cáo.

In [ ]:
from src.metrics import recognition_metrics  # bản chuẩn trong repository

metrics = {
    "model": MODEL_NAME,
    "split": "rec_test.txt",
    **recognition_metrics(prediction_rows),
    "batch_size": BATCH_SIZE,
    "model_init_seconds": model_init_seconds,
    "prediction_seconds": prediction_seconds,
    "environment": ENV_INFO,
}

METRICS_PATH = RESULTS_DIR / "metrics_baseline_ppocrv5_mobile.json"
METRICS_PATH.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Metrics:", METRICS_PATH)

## 10. Kiểm tra lỗi và lấy output từ Kaggle

Cell cuối in tối đa 20 mẫu dự đoán sai đầu tiên để kiểm tra nhanh. Sau khi chạy xong, tải file JSONL và metrics JSON từ Kaggle Output; sau đó copy chúng về repository local, kiểm tra rồi commit nếu dung lượng phù hợp. Không push trực tiếp từ Kaggle.

In [ ]:
incorrect_rows = [row for row in prediction_rows if normalize_text(row["gt"]) != normalize_text(row["pred"])]

print(f"Incorrect predictions: {len(incorrect_rows):,} / {len(prediction_rows):,}")
for row in incorrect_rows[:20]:
    print(json.dumps(row, ensure_ascii=False))

print("Files to download:")
for path in [PREDICTIONS_PATH, METRICS_PATH]:
    print(f"{path} ({path.stat().st_size / 1024:.1f} KB)")